[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akifkoc/sabanci-qiskitfallfest26/blob/main/1-kubit.ipynb)

# Bölüm 1: Bitten Kubite

**Qiskit ile Kuantum Programlamaya Giriş** · Qiskit Fall Fest 2026 · Sabancı Üniversitesi

Bu bölümde klasik bitten kubite (*qubit*) geçeceğiz. Gereken doğrusal cebiri (*linear algebra*) **Doğrusal cebir** kutularında kısaca tanıtacağız. Bölümün sonunda şunları yapabileceksiniz:

- bitleri ve kubitleri vektörlerle, işlemleri matrislerle göstermek,
- bir kubitin ölçüm olasılıklarını hesaplamak,
- Qiskit ile X, Z ve H kapılarını kullanan devreler kurup çalıştırmak,
- girişimin (*interference*) ne olduğunu açıklamak.

**Nasıl kullanılır?** Hücreleri sırayla çalıştırın (Shift + Enter). **Soru**ları önce kendiniz düşünün, sonra "Cevap"a tıklayın. **Görev** hücrelerinde yorum satırlarındaki yönergeleri izleyin. *Ek görev* isteğe bağlıdır. Takıldığınız yerde bize sorun.

In [ ]:
# Google Colab'da bu hücreyi bir kez çalıştırın (yaklaşık 1 dakika sürer).
# Kendi bilgisayarınızda requirements.txt ile kurulum yaptıysanız bu hücreyi atlayabilirsiniz.
%pip install -q "qiskit[visualization]==2.6.0" "qiskit-aer==0.18.0"

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

simulator = AerSimulator()   # kuantum devrelerini bilgisayarımızda çalıştıran simülatör

## 1. Klasik bit

Bir bit her an iki durumdan birindedir: $0$ ya da $1$. Bir bit üzerinde dört işlem yapabiliriz:

1. Birim (*identity*): $I(0) = 0$, $I(1) = 1$
2. Değil (*NOT*): $\mathrm{DE\breve{G}\dot{I}L}(0) = 1$, $\mathrm{DE\breve{G}\dot{I}L}(1) = 0$
3. Sabit sıfır (*ZERO*): $\mathrm{SIFIR}(0) = 0$, $\mathrm{SIFIR}(1) = 0$
4. Sabit bir (*ONE*): $\mathrm{B\dot{I}R}(0) = 1$, $\mathrm{B\dot{I}R}(1) = 1$

**Soru:** Hangi işlemlerden sonra, sonuca bakarak başlangıç değerini bulabiliriz?

<details>
<summary><b>Cevap</b></summary>

$I$ ve $\mathrm{DE\breve{G}\dot{I}L}$: farklı başlangıçlar farklı sonuçlara gider. Bunlar **tersinir** (*reversible*) işlemlerdir. $\mathrm{SIFIR}$ ve $\mathrm{B\dot{I}R}$ ise iki başlangıcı da aynı sonuca götürür; bunlar **tersinmezdir** (*irreversible*). Kuantum kapıları her zaman tersinirdir.

</details>

### Vektör gösterimi

$$ 0 = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \qquad 1 = \begin{pmatrix} 0 \\ 1 \end{pmatrix} $$

Her işlem bir matristir. Sütunlar başlangıç durumlarına, satırlar son durumlara karşılık gelir:

$$ I = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}, \quad \mathrm{DE\breve{G}\dot{I}L} = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad \mathrm{SIFIR} = \begin{pmatrix} 1 & 1 \\ 0 & 0 \end{pmatrix}, \quad \mathrm{B\dot{I}R} = \begin{pmatrix} 0 & 0 \\ 1 & 1 \end{pmatrix} $$

> **Doğrusal cebir: matris-vektör çarpımı**
>
> Sonucun her elemanı, matrisin bir satırıyla vektörün çarpımıdır: satırdaki sayıları vektördeki karşılıklarıyla çarpıp toplarız.
>
> $$ \begin{pmatrix} a & b \\ c & d \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix} = \begin{pmatrix} a x + b y \\ c x + d y \end{pmatrix} $$

Python'da matris çarpımı için `@` işleci kullanılır. Bu işleci kullanmak için matris ve vektörleri `np.array` ile tanımlamamız gerekir. Kodda işlemlerin İngilizce adlarını kullanıyoruz: $\mathrm{DE\breve{G}\dot{I}L}$ için `NOT`.

In [ ]:
state_0 = np.array([1, 0])   # 0 durumu
state_1 = np.array([0, 1])   # 1 durumu

NOT = np.array([[0, 1],
                [1, 0]])     # DEĞİL matrisi

print("NOT · 0 =", NOT @ state_0)
print("NOT · 1 =", NOT @ state_1)

**Soru:** Birisi $\mathrm{SIFIR}$ matrisini $\begin{pmatrix} 0 & 0 \\ 0 & 0 \end{pmatrix}$ diye yazmış. Neden yanlış?

<details>
<summary><b>Cevap</b></summary>

Bu matris her durumu $\begin{pmatrix} 0 \\ 0 \end{pmatrix}$'a, yani **sıfır vektörüne** götürür; sıfır vektörü hiçbir bit durumu değildir. $\mathrm{SIFIR}$'ın sonucu $0$ durumu, yani $\begin{pmatrix} 1 \\ 0 \end{pmatrix}$ olmalı: doğru matris $\begin{pmatrix} 1 & 1 \\ 0 & 0 \end{pmatrix}$'dır.

</details>

## 2. Olasılıksal bit

Bir yazı-tura atıp sonuca bakmazsak bilgimiz olasılıksaldır. Yazıyı $0$, turayı $1$ ile gösterelim. **Olasılıksal bit** (*probabilistic bit*):

$$ \begin{pmatrix} a \\ b \end{pmatrix} = a \begin{pmatrix} 1 \\ 0 \end{pmatrix} + b \begin{pmatrix} 0 \\ 1 \end{pmatrix}, \qquad a, b \geq 0, \quad a + b = 1 $$

$a$, $0$'ın; $b$, $1$'in olasılığıdır. Örneğin hilesiz para: $\begin{pmatrix} 1/2 \\ 1/2 \end{pmatrix}$.

> **Doğrusal cebir: doğrusal bileşim ve $\ell_1$ normu**
>
> Vektörleri sayılarla çarpıp toplamaya **doğrusal bileşim** (*linear combination*) denir. Elemanların mutlak değerlerinin toplamı $|a| + |b|$, vektörün **$\ell_1$ normudur**. Olasılıksal bitin $\ell_1$ normu her zaman $1$'dir. İki nokta arasındaki farkın $\ell_1$ normu, **Manhattan uzaklığıdır** (*Manhattan distance*): ızgara biçimindeki sokaklarda gidilebilecek en kısa yol. Örneğin $(0, 0)$'dan $(3, 4)$'e: $3 + 4 = 7$.

Olasılıksal bir işlem de bir matristir; elemanları negatif değildir ve her sütununun toplamı $1$'dir (**stokastik matris**). Hilesiz para atmak:

$$ \mathrm{AD\dot{I}L} = \begin{pmatrix} \frac{1}{2} & \frac{1}{2} \\ \frac{1}{2} & \frac{1}{2} \end{pmatrix} $$

**Soru:** Yazı ile başlayıp hilesiz parayı **iki kez** atarsak durum ne olur? Tahmin edin, sonra aşağıdaki hücreyi çalıştırın.

<details>
<summary><b>Cevap</b></summary>

Yine $\begin{pmatrix} 1/2 \\ 1/2 \end{pmatrix}$. Birinci atıştan sonra durum yarı yarıyadır; ikinci atış bunu değiştirmez. Birazdan bunun kuantum dünyasında geçerli olmadığını göreceğiz.

</details>

In [ ]:
FAIR = np.array([[0.5, 0.5],
                 [0.5, 0.5]])   # ADİL matrisi

start = np.array([1, 0])        # başlangıç: yazı (0)
print("bir atıştan sonra:", FAIR @ start)
print("iki atıştan sonra:", FAIR @ FAIR @ start)

## 3. Bir deney: kuantum yazı-tura

*Deney ve görseller QWorld'ün QBook101 kitabındaki "Quantum Coin Flipping" not defterinden alınmıştır (CC BY 4.0).*

Bir lazer ışınını **ışın bölücüye** (*beam splitter*) gönderelim: ışığın yarısı geçer, yarısı yansır. Işığı tek tek **fotonlara** inene kadar zayıflatalım.

**Soru:** Tek bir foton ışın bölücüye gelirse ne olur? İkiye mi bölünür, yoksa bir yolu mu seçer?

<details>
<summary><b>Cevap</b></summary>

Bölünmez. Her foton bütün hâlinde tek bir dedektörde görülür; iki dedektör neredeyse hiçbir zaman aynı anda tıklamaz (Grangier, Roger ve Aspect, 1986). Çok sayıda fotonla iki dedektör de fotonların yaklaşık yarısını görür: ışın bölücü hilesiz bir para gibidir.

<img src="https://gitlab.com/qworld/qeducation/qbook101/raw/main/qbook101/images/ch2/photon/photon2.jpg" alt="Işın bölücü ve iki dedektör: yaklaşık yüzde 50 - yüzde 50" width="45%">

</details>

Şimdi iki ayna ve ikinci bir ışın bölücü ekleyelim. Bu düzeneğe **Mach–Zehnder girişimölçeri** (*Mach–Zehnder interferometer*) denir.

<img src="https://gitlab.com/qworld/qeducation/qbook101/raw/main/qbook101/images/ch2/photon/photon6.jpg" alt="Mach–Zehnder girişimölçeri: B1 ve B2 dedektörleri" width="50%">

**Soru:** Foton her ışın bölücüde bir yazı-tura atıyorsa, B1 ve B2 fotonların yüzde kaçını görür?

<details>
<summary><b>Cevap</b></summary>

Yazı-tura modeline göre $\%50$ ve $\%50$ (yukarıdaki iki atış hesabı). **Ama deneyde fotonların hepsi B1'e gidiyor!**

<img src="https://gitlab.com/qworld/qeducation/qbook101/raw/main/qbook101/images/ch2/photon/photon7.jpg" alt="Deneyin sonucu: B1 yüzde 100, B2 yüzde 0" width="50%">

Lazer ışığında bunu dalgalar açıklar: iki yoldan gelen dalgalar B1'de birbirini güçlendirir, B2'de söndürür. Şaşırtıcı olan, **bölünmeyen tek fotonların** da böyle davranması: iki yol da sonucu etkiliyor. Olasılıklar negatif olamadığı için birbirini söndüremez; yazı-tura modeli bunu açıklayamaz. Kuantum mekaniğine ihtiyacımız var.

</details>

## 4. Kubit

**Kubitin** (*qubit*) iki baz durumu (*basis state*) vardır:

$$ |0\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \qquad |1\rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix} $$

Bir kubit bu iki durumun bir doğrusal bileşiminde bulunabilir. Buna **üst üste binme** (süperpozisyon, *superposition*) denir:

$$ |\psi\rangle = \alpha\,|0\rangle + \beta\,|1\rangle = \begin{pmatrix} \alpha \\ \beta \end{pmatrix} $$

$\alpha$ ve $\beta$'ya **genlik** (*amplitude*) denir. Genlikler **negatif** olabilir.

**Ölçüm** (*measurement*): $0$'ı $\alpha^2$, $1$'i $\beta^2$ olasılıkla görürüz. Ölçümden sonra kubit gördüğümüz duruma **çöker** (*collapse*).

> **Doğrusal cebir: $\ell_2$ normu**
>
> Olasılıkların toplamı $1$ olduğundan $\alpha^2 + \beta^2 = 1$: vektörün **$\ell_2$ normu**, yani uzunluğu $1$'dir. (Olasılıksal bitte $\ell_1$ normu $1$'di.) İki nokta arasındaki farkın $\ell_2$ normu kuş uçuşu uzaklıktır: $(0, 0)$'dan $(3, 4)$'e $\sqrt{3^2 + 4^2} = 5$. Genlikler karmaşık sayı olduğunda koşul $|\alpha|^2 + |\beta|^2 = 1$ olur.

**Soru:** $\begin{pmatrix} 3/5 \\ -4/5 \end{pmatrix}$ durumundaki kubiti ölçersek $0$'ı ve $1$'i hangi olasılıklarla görürüz?

<details>
<summary><b>Cevap</b></summary>

$0$: $\left(\frac35\right)^2 = \frac{9}{25}$, $1$: $\left(-\frac45\right)^2 = \frac{16}{25}$. Toplam $1$. Eksi işareti olasılığı değiştirmez.

</details>

## 5. Qiskit ile ilk kuantum devremiz

Qiskit, IBM'in açık kaynaklı kuantum programlama kütüphanesidir. Bir **kuantum devresi** (*quantum circuit*) kurarız: kubitler $|0\rangle$'dan başlar, **kapılar** (*gates*) uygularız, ölçeriz ve devreyi birçok kez çalıştırıp sonuçları sayarız. Kubitler $0$'dan başlayarak numaralanır.

### X kapısı

$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$, yani $\mathrm{DE\breve{G}\dot{I}L}$'in kuantum karşılığı: $X|0\rangle = |1\rangle$, $X|1\rangle = |0\rangle$.

In [ ]:
qc = QuantumCircuit(1, 1)   # 1 kubit, 1 klasik bit

qc.x(0)            # 0 numaralı kubite X kapısı
qc.measure(0, 0)   # 0 numaralı kubiti ölç, sonucu 0 numaralı klasik bite yaz

qc.draw("mpl")

In [ ]:
# devreyi 1000 kez çalıştırıp ölç (shots: tekrar sayısı) ve sonuçları say
counts = simulator.run(qc, shots=1000).result().get_counts()
print(counts)

Her çalıştırıp ölçmeye bir **tekrar** (*shot*) diyoruz. Sonuç bir **sözlüktür** (*dictionary*): `{'1': 1000}`, 1000 tekrarın hepsinde $1$ gördük demek.

### Z kapısı

$Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$: $Z|0\rangle = |0\rangle$, $Z|1\rangle = -|1\rangle$.

### Görev 1

Önce X, sonra Z uygulayıp ölçün. Sonucu tahmin edin, sonra çalıştırın.

In [ ]:
qc = QuantumCircuit(1, 1)

# kodunuzu buraya yazın (önce X, sonra Z)

qc.measure(0, 0)
counts = simulator.run(qc, shots=1000).result().get_counts()
print(counts)

### Hadamard kapısı (H)

$$ H = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}, \qquad H|0\rangle = \frac{1}{\sqrt{2}}|0\rangle + \frac{1}{\sqrt{2}}|1\rangle = |+\rangle, \qquad H|1\rangle = \frac{1}{\sqrt{2}}|0\rangle - \frac{1}{\sqrt{2}}|1\rangle = |-\rangle $$

İki durumda da $0$ ve $1$'i $\frac12$ olasılıkla görürüz: H bir **kuantum yazı-turadır**, yani ışın bölücü.

In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)

counts = simulator.run(qc, shots=1000).result().get_counts()
print(counts)
plot_histogram(counts)

**Durum vektörü** (*state vector*): Gerçek kuantum bilgisayarında genlikleri göremeyiz, yalnızca ölçüm sonuçlarını görürüz. Bilgisayarda yaptığımız simülasyonda ise genlikleri `Statevector` ile okuyabiliriz (ölçüm eklemeden). Çıktıdaki $\frac{\sqrt{2}}{2}$, $\frac{1}{\sqrt{2}}$ ile aynıdır.

In [ ]:
qc = QuantumCircuit(1)   # ölçüm yok
qc.h(0)

Statevector(qc).draw("latex")

### Görev 2

Önce X, sonra H uygulayın. Durum vektörüne ve ölçüm sonuçlarına bakın. $|+\rangle$ ile $|-\rangle$'yi ölçerek ayırt edebilir miyiz? (`measure_all()` tüm kubitleri ölçer.)

In [ ]:
qc = QuantumCircuit(1)

# kodunuzu buraya yazın (önce X, sonra H)

display(Statevector(qc).draw("latex"))

qc.measure_all()
counts = simulator.run(qc, shots=1000).result().get_counts()
print(counts)

## 6. Girişim

Mach–Zehnder deneyini iki H kapısıyla yapalım:

In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.h(0)
qc.measure(0, 0)

counts = simulator.run(qc, shots=1000).result().get_counts()
print(counts)
qc.draw("mpl")

**Soru:** Neden hep $0$? İpucu: İkinci H'yi $\frac{1}{\sqrt{2}}|0\rangle$ ve $\frac{1}{\sqrt{2}}|1\rangle$ parçalarına ayrı ayrı uygulayın.

<details>
<summary><b>Cevap</b></summary>

$\frac{1}{\sqrt{2}}H|0\rangle = \frac12|0\rangle + \frac12|1\rangle$ ve $\frac{1}{\sqrt{2}}H|1\rangle = \frac12|0\rangle - \frac12|1\rangle$. Toplayınca $|0\rangle$'ın genlikleri birbirini güçlendirir ($\frac12 + \frac12 = 1$), $|1\rangle$'inkiler birbirini yok eder ($\frac12 - \frac12 = 0$).

Buna **girişim** (*interference*) denir: aynı işaretli katkılar **yapıcı**, zıt işaretli katkılar **yıkıcı** girişim yapar. Olasılıklar negatif olamadığı için klasik yazı-turada yıkıcı girişim yoktur. (Parçalara ayrı ayrı uygulayabildik, çünkü matrisle çarpma **doğrusaldır**.)

</details>

### Ek görev: X, H, H

Önce X, sonra iki kez H uygulayıp ölçün. Sonucu tahmin edin, sonra çalıştırın.

In [ ]:
qc = QuantumCircuit(1, 1)

# kodunuzu buraya yazın (X, H, H)

qc.measure(0, 0)
counts = simulator.run(qc, shots=1000).result().get_counts()
print(counts)

### Görev 3

Sırasıyla H, Z ve H uygulayıp ölçün. Z kapısının etkisini artık görebiliyor muyuz?

In [ ]:
qc = QuantumCircuit(1, 1)

# kodunuzu buraya yazın (H, Z, H)

qc.measure(0, 0)
counts = simulator.run(qc, shots=1000).result().get_counts()
print(counts)

> **Doğrusal cebir: matris çarpımı**
>
> Önce $B$, sonra $A$ kapısını uygulamak, $AB$ matrisini uygulamakla aynıdır: $A(B\,v) = (AB)\,v$. Devre soldan sağa okunur, matrisler sağdan sola yazılır: H, Z, H devresinin matrisi $HZH$'dir. Sırası genellikle önemlidir: $XZ \neq ZX$.
>
> Kuantum kapıları vektörün uzunluğunu ($\ell_2$ normunu) korur; böyle matrislere **birimsel matris** (*unitary matrix*; Türkçe kaynaklarda *üniter matris*) denir. (Birim matris $I$ ile karıştırmayın: $X$, $Z$, $H$ birimseldir ama birim matris değildir.)

In [ ]:
X = np.array([[0, 1], [1, 0]])
Z = np.array([[1, 0], [0, -1]])
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)

print("H H =");   print(np.round(H @ H, 3) + 0.0)       # birim matris
print("H Z H ="); print(np.round(H @ Z @ H, 3) + 0.0)   # X matrisi (Görev 3)

> **Doğrusal cebir: iç çarpım ve diklik**
>
> İki gerçel vektörün **iç çarpımı** (*inner product*): $\langle u | v \rangle = u_0 v_0 + u_1 v_1$. İç çarpımı $0$ olan vektörler birbirine **diktir** (*orthogonal*): örneğin $|0\rangle$ ile $|1\rangle$, $|+\rangle$ ile $|-\rangle$. Dik durumlar doğru ölçümle kesin olarak ayırt edilebilir (Görev 3).

**Soru:** İki H kapısının **arasında** kubiti ölçersek sonuç ne olur?

<details>
<summary><b>Cevap</b></summary>

Ara ölçüm kubiti $|0\rangle$ ya da $|1\rangle$'e çöktürür; ikinci H yeniden yazı-tura atar. Girişim kaybolur, sonuç yarı yarıya olur.

</details>

## Özet

- Bit ve olasılıksal bit vektörlerle, işlemler matrislerle gösterilir.
- Kubit $\alpha|0\rangle + \beta|1\rangle$ durumundadır; genlikler negatif olabilir, ölçüm olasılıkları genliklerin kareleridir.
- X, Z ve H birer birimsel matristir; Qiskit'te `qc.x`, `qc.z`, `qc.h` ile uygulanır.
- Negatif genlikler girişime yol açar.

**Sıradaki bölüm:** [Bölüm 2: Birden Fazla Kubit](https://colab.research.google.com/github/akifkoc/sabanci-qiskitfallfest26/blob/main/2-cok_kubitli_sistemler.ipynb) (birden fazla kubit ve CNOT kapısı).

### Daha fazlası

- Qiskit: [ibm.com/quantum/qiskit](https://www.ibm.com/quantum/qiskit)
- IBM Quantum Learning: [quantum.cloud.ibm.com/learning](https://quantum.cloud.ibm.com/learning/en)
- QWorld QBook101, Bölüm 2: [qworld.net/qbook101](https://qworld.net/qbook101/)
- QTurkey (Kuantum Türkiye) etkinlikleri: [qturkey.org/events](https://qturkey.org/events/)